# Week 5 — GroupBy & Aggregation: GroupBy Fundamentals
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Use `.groupby()` with single and multiple keys
- Apply aggregation functions: `sum`, `mean`, `count`, `nunique`, `min`, `max`
- Use `.agg()` for multiple aggregations at once
- Reset the index after a groupby so the result is a normal DataFrame

---

## 2. Why This Session Matters

Olist has **99,441 orders** spread across **27 Brazilian states** and dozens of product categories. Without grouping, you can only inspect one row at a time — useless when a manager asks *"which state generates the most revenue?"*

GroupBy answers exactly that kind of question in a single line of code. It is the programming version of an Excel pivot table: split the data into groups, apply a calculation to each group, and combine the results back together. Today you'll move from looking at individual orders to summarising the entire business — orders per state, revenue per seller, items per order — the questions analysts actually get paid to answer.

## 3. Setup

We mount Google Drive and load only the three tables we need today: **orders**, **customers**, and **order items**. Each `print` confirms the row count so you know the data loaded correctly before we start grouping.

In [ ]:
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')
olist_path = '/content/drive/MyDrive/olist-data'

orders = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(olist_path, 'olist_customers_dataset.csv'))
items = pd.read_csv(os.path.join(olist_path, 'olist_order_items_dataset.csv'))

print(f"✓ orders: {len(orders):,} rows")
print(f"✓ customers: {len(customers):,} rows")
print(f"✓ items: {len(items):,} rows")
# Expected: orders: 99,441 rows | customers: 99,441 rows | items: 112,650 rows

## 4. Concept 1 — Single-Key GroupBy

**The split-apply-combine model.** A groupby does three things:
1. **Split** the rows into groups that share the same value in a column (e.g. all `delivered` orders together).
2. **Apply** a function to each group (e.g. `count`, `sum`, `mean`).
3. **Combine** the per-group answers into a single result.

Think of it as an Excel pivot table written in code: the groupby column is the "Rows" area and the aggregation is the "Values" area. The pattern is always:

```python
df.groupby('group_column')['value_column'].agg_function()
```

Let's start by counting how many orders fall under each `order_status`.

In [ ]:
# Split by order_status, apply count to the order_id column
status_counts = orders.groupby('order_status')['order_id'].count()
print(status_counts.sort_values(ascending=False))

print(f"\nDelivered orders: {status_counts['delivered']:,}")
print(f"Shipped orders:   {status_counts['shipped']:,}")
# Expected: delivered = 96,478 | shipped = 1,107

The `order_status` lives in `orders`, but the customer's **state** lives in the `customers` table. To group orders by state we first **merge** the two tables on `customer_id`, then group the merged result by `customer_state`.

In [ ]:
# Merge orders with customers so every order carries its customer_state
orders_customers = orders.merge(customers, on='customer_id')

state_orders = orders_customers.groupby('customer_state')['order_id'].count()
print(state_orders.sort_values(ascending=False).head(5))
# Expected:
# SP    41746
# RJ    12852
# MG    11635
# RS     5466
# PR     5045

print(f"\nSP orders: {state_orders['SP']:,}")
# Expected: SP orders: 41,746

Order *counts* tell us where customers are, but managers care about **revenue**. The price of each item lives in the `items` table, so we merge once more (orders+customers already merged, now add items on `order_id`) and sum the `price` per state.

In [ ]:
# Add item-level prices, then sum price per state
orders_items = orders_customers.merge(items, on='order_id')

state_revenue = orders_items.groupby('customer_state')['price'].sum()
print(state_revenue.sort_values(ascending=False).head(5))
# Expected:
# SP    5202955.05
# RJ    1824092.67
# MG    1585308.03
# RS     750304.02
# PR     683083.76

print(f"\nSP revenue: R${state_revenue['SP']:,.2f}")
# Expected: SP revenue: R$5,202,955.05

## 5. Concept 2 — `.agg()` for Multiple Metrics

So far each groupby produced **one** number per group. But a real seller report needs several metrics at once: total revenue, average price, number of items, number of distinct orders. Running four separate groupbys is slow and clumsy.

`.agg()` with **named aggregations** computes them all in one pass. The syntax is:

```python
df.groupby('key').agg(
    new_column_name=('source_column', 'function')
).reset_index()
```

The `.reset_index()` turns the grouped key back into a normal column so the result is a tidy DataFrame you can sort, filter, or chart.

In [ ]:
# One pass, four metrics per seller
seller_stats = items.groupby('seller_id').agg(
    total_revenue=('price', 'sum'),
    avg_price=('price', 'mean'),
    total_items=('order_item_id', 'count'),
    unique_orders=('order_id', 'nunique')
).reset_index()

print(seller_stats.nlargest(5, 'total_revenue'))

top_revenue = seller_stats['total_revenue'].max()
print(f"\nTop seller revenue: R${top_revenue:,.2f}")
# Expected: Top seller revenue: R$229,472.63

The same pattern works when we group by `order_id`. This builds an **order-level summary** — collapsing the multiple item rows of an order into one row with the total price, total freight, and item count. This is how we discover how many orders contain more than one item.

In [ ]:
# Collapse item rows into one summary row per order
order_summary = items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    item_count=('order_item_id', 'count')
).reset_index()

print(order_summary.head())

print(f"\nOrders with >1 item: {(order_summary['item_count'] > 1).sum():,}")
# Expected: Orders with >1 item: 9,803
print(f"Max items in one order: {order_summary['item_count'].max()}")
# Expected: Max items in one order: 21
print(f"Avg items per order: {order_summary['item_count'].mean():.2f}")
# Expected: Avg items per order: 1.14

### Going Deeper — Aggregation with Transform and Filter

Two powerful extensions of groupby:

- **`.transform()`** returns a result *the same length as the original DataFrame*, so you can attach a group-level number back onto every row (e.g. each state's total revenue beside every individual order). Unlike `.agg()`, it does **not** collapse the rows.
- **`.filter()`** keeps only the rows that belong to groups passing a test (e.g. only states with more than 10,000 orders).

These let you ask "how does this row compare to its group?" — something a plain aggregation can't answer.

In [ ]:
# transform: attach each state's total revenue onto every order row (no collapsing)
orders_items['state_total_revenue'] = (
    orders_items.groupby('customer_state')['price'].transform('sum')
)
print(orders_items[['customer_state', 'price', 'state_total_revenue']].head(3))
print(f"\nRows after transform: {len(orders_items):,} (same as before — nothing collapsed)")
# Expected: every SP row shows state_total_revenue = 5202955.05

# filter: keep only orders belonging to states with more than 10,000 orders
big_states = orders_customers.groupby('customer_state').filter(
    lambda g: len(g) > 10000
)
print(f"\nStates with >10,000 orders: {sorted(big_states['customer_state'].unique())}")
# Expected: ['MG', 'RJ', 'SP']  (SP=41,746, RJ=12,852, MG=11,635)


### Common Mistakes

Three groupby traps that catch almost everyone at first. Read the `# ❌` line, understand *why* it bites, then use the `# ✅` version.

In [ ]:
# Mistake 1 — forgetting .reset_index() leaves a Series with the group as its index
# ❌ Wrong — the result is a Series, not a clean DataFrame; column names look confusing later
result = orders.groupby('order_status')['order_id'].count()
print(type(result))            # <class 'pandas.core.series.Series'>

# ✅ Correct — .reset_index() gives a proper two-column DataFrame you can sort/merge/chart
result = orders.groupby('order_status')['order_id'].count().reset_index()
result.columns = ['order_status', 'count']
print(type(result))           # <class 'pandas.core.frame.DataFrame'>
print(result[result['order_status'] == 'delivered'])
# Expected: delivered  96478

# Mistake 2 — grouping on a column that lives in another table
# ❌ Wrong — 'customer_state' is NOT in orders, so this raises KeyError
#    orders.groupby('customer_state')['order_id'].count()
# ✅ Correct — merge customers in FIRST, then group
ok = orders.merge(customers, on='customer_id').groupby('customer_state')['order_id'].count()
print(f"\nSP after merging first: {ok['SP']:,}")
# Expected: SP after merging first: 41,746

# Mistake 3 — counting rows when you meant distinct orders
# ❌ count() counts item ROWS (112,650) — inflated because an order can have many items
# ✅ nunique() counts distinct orders
print(f"\nitem rows (count):     {items.groupby('seller_id')['order_id'].count().sum():,}")
print(f"distinct orders (nunique): {items['order_id'].nunique():,}")
# Expected: item rows: 112,650 | distinct orders: 98,666


## ⏱ Mini Challenge (5 minutes)

Your turn — quick one before the group exercise. Using the data already loaded:

- Which **state** has the highest **average order item price** (`price`)?
- **Hint:** start from `orders_items` (already merged: orders + customers + items), `groupby('customer_state')`, take the **mean** of `price`, then sort descending and look at the top row.
- Sanity check: SP is the *biggest* state by volume — but is it the most *expensive* per item? Find out.

In [ ]:
# ⏱ Mini Challenge — highest average item price by state
# YOUR CODE HERE
# Hint: orders_items.groupby('customer_state')['price'].mean().sort_values(ascending=False).head()



## 7. Group Exercise (40 min)

Work in your breakout groups. Each task has a **verified expected value** — your code is correct when it reproduces it.

1. **Orders per state.** Using the merged `orders_customers`, count orders per `customer_state` and print the top 5. *Verify SP = 41,746.*
2. **Seller revenue.** Using `items`, compute total revenue and average price by `seller_id`, then print the top 5 by revenue. *Verify the top seller = R$229,472.63.*
3. **Order summary.** Group `items` on `order_id` to build `order_summary` with `total_price`, `total_freight`, and `item_count`. *Verify max item count = 21 and orders with >1 item = 9,803.*
4. **DeepSeek task.** Ask DeepSeek to write groupby code that, for each `order_status`, returns the **count of orders**, the **percentage of all orders**, and the **average approval delay** (`order_approved_at` minus `order_purchase_timestamp`). Paste its code below, run it, and validate that the delivered count = 96,478.

In [ ]:
# ===== Task 1: Orders per state (top 5, verify SP = 41,746) =====
# Hint: groupby 'customer_state' on orders_customers, count 'order_id'
# YOUR CODE HERE


# ===== Task 2: Total revenue + avg price by seller (top 5) =====
# Hint: items.groupby('seller_id').agg(total_revenue=..., avg_price=...)
# Top seller revenue should be R$229,472.63
# YOUR CODE HERE


# ===== Task 3: Build order_summary (verify max_items=21, >1 item orders=9,803) =====
# Hint: items.groupby('order_id').agg(total_price=..., total_freight=..., item_count=...)
# YOUR CODE HERE


# ===== Task 4: DeepSeek — order_status count, percentage, avg approval delay =====
# Paste DeepSeek's code here, then validate delivered count = 96,478
# YOUR CODE HERE


## 8. What We Covered Today

| Concept | Pandas Method | Key Result |
|---|---|---|
| Count by group | `.groupby().count()` | delivered: 96,478 |
| Sum by group | `.groupby().sum()` | SP revenue: R$5,202,955.05 |
| Multiple metrics | `.agg()` | Top seller: R$229,472.63 |
| Attach group value to every row | `.transform()` | rows unchanged |
| Keep only qualifying groups | `.filter()` | states >10k: SP, RJ, MG |

**You also learned to dodge the three classic traps:** forgetting `.reset_index()`, grouping on a column that lives in another table, and using `count()` (rows) when you meant `nunique()` (distinct orders).

## Coming Up Thursday

In Thursday's session we'll explore **time series groupby** — parsing dates with `pd.to_datetime()` and finding patterns like "which month had the most orders?" (spoiler: **November 2017, Black Friday!**). We'll also do **multi-key groupby** using two columns at once to build cross-tabulations like orders per state per year.

See you Thursday! 📊